In [1]:
# %% [markdown]
"""
# Contact Planes and Misorientation Analysis
This notebook performs detailed grain boundary analysis using latvec outputs.
"""

# %%
# Import necessary modules
import numpy as np
from pathlib import Path
import sys

# Add src to path if needed
sys.path.append('/home/sgarg/structural_analysis_gb')  # Uncomment and modify if needed

from src.contactplanes import (
    read_grain_vectors,
    assign_abc,
    compute_com_from_gro,
    normalize,
    contactplan
)
from src.misorientation_angle import (
    misorientation_for_group,
    unit
)

In [2]:
# %% [markdown]
"""
## Configuration: Set Your File Paths
"""

# %%
# ==== MODIFY THESE PATHS ====
g1_gro_file = "/home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g1.gro"  # Grain 1 .gro file
g2_gro_file = "/home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g2.gro"  # Grain 2 .gro file

# Option 1: Auto-detect output.txt files (assumes <stem>_latvecs.txt naming)
# g1_txt = None  # Will auto-detect
# g2_txt = None  # Will auto-detect

# Option 2: Manually specify output.txt paths (uncomment to use)
g1_txt = "/home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g1_latvecs.txt"
g2_txt = "/home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g2_latvecs.txt"

# Symmetry for misorientation (default: triclinic for pentacene)
symmetry_name = "triclinic"  # Options: "triclinic", "monoclinic_b", "orthorhombic", etc.

print(f"Grain 1 GRO: {g1_gro_file}")
print(f"Grain 2 GRO: {g2_gro_file}")
print(f"Symmetry: {symmetry_name}")

Grain 1 GRO: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g1.gro
Grain 2 GRO: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g2.gro
Symmetry: triclinic


In [3]:
"""
## 1. Contact Plane Analysis
"""

# %%
print("="*80)
print("CONTACT PLANE ANALYSIS")
print("="*80)

# Determine latvec output file paths
if g1_txt is None:
    g1_txt = Path(g1_gro_file).with_name(f"{Path(g1_gro_file).stem}_latvecs.txt")
if g2_txt is None:
    g2_txt = Path(g2_gro_file).with_name(f"{Path(g2_gro_file).stem}_latvecs.txt")

print(f"\nGrain 1 latvec file: {g1_txt}")
print(f"Grain 2 latvec file: {g2_txt}")

# Read grain vectors from output.txt files
ff1, ef1 = read_grain_vectors(g1_txt)
ff2, ef2 = read_grain_vectors(g2_txt)

print("\n" + "-"*80)
print("GRAIN 1 - Raw Latvec Vectors")
print("-"*80)
print(f"ff vector: {ff1}")
print(f"ef vector: {ef1}")

print("\n" + "-"*80)
print("GRAIN 2 - Raw Latvec Vectors")
print("-"*80)
print(f"ff vector: {ff2}")
print(f"ef vector: {ef2}")

# Assign a, b, c axes
a1, b1, c1 = assign_abc(ff1, ef1)
a2, b2, c2 = assign_abc(ff2, ef2)

print("\n" + "="*80)
print("GRAIN 1 - Normalized Lattice Axes (a, b, c)")
print("="*80)
print(f"a1 (normalized ff): {a1}")
print(f"b1 (normalized ef): {b1}")
print(f"c1 (a × b):         {c1}")
print(f"\nMagnitudes check:")
print(f"  |a1| = {np.linalg.norm(a1):.6f}")
print(f"  |b1| = {np.linalg.norm(b1):.6f}")
print(f"  |c1| = {np.linalg.norm(c1):.6f}")

print("\n" + "="*80)
print("GRAIN 2 - Normalized Lattice Axes (a, b, c)")
print("="*80)
print(f"a2 (normalized ff): {a2}")
print(f"b2 (normalized ef): {b2}")
print(f"c2 (a × b):         {c2}")
print(f"\nMagnitudes check:")
print(f"  |a2| = {np.linalg.norm(a2):.6f}")
print(f"  |b2| = {np.linalg.norm(b2):.6f}")
print(f"  |c2| = {np.linalg.norm(c2):.6f}")

CONTACT PLANE ANALYSIS

Grain 1 latvec file: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g1_latvecs.txt
Grain 2 latvec file: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g2_latvecs.txt

--------------------------------------------------------------------------------
GRAIN 1 - Raw Latvec Vectors
--------------------------------------------------------------------------------
ff vector: [-0.731087 -0.04657   0.680693]
ef vector: [-0.412071  0.782957  0.466021]

--------------------------------------------------------------------------------
GRAIN 2 - Raw Latvec Vectors
--------------------------------------------------------------------------------
ff vector: [-0.987355 -0.079759  0.136997]
ef vector: [-0.528447  0.840277 -0.121156]

GRAIN 1 - Normalized Lattice Axes (a, b, c)
a1 (normalized ff): [-0.73108703 -0.04657     0.68069302]
b1 (normalized ef): [-0.41207105  0.7829571   0.46602106]
c1 (a × b):         [-0.68208423  0.07404

In [4]:
# %%
# Compute centers of mass and contact vector
com1 = compute_com_from_gro(g1_gro_file)
com2 = compute_com_from_gro(g2_gro_file)

print("\n" + "="*80)
print("CENTER OF MASS & CONTACT VECTOR")
print("="*80)
print(f"\nGrain 1 COM: {com1}")
print(f"Grain 2 COM: {com2}")

conn_vec = com2 - com1
contact_vec = normalize(conn_vec)

print(f"\nConnection vector (COM2 - COM1): {conn_vec}")
print(f"Distance between COMs: {np.linalg.norm(conn_vec):.4f} Å")
print(f"\nNormalized contact vector: {contact_vec}")
print(f"Magnitude check: {np.linalg.norm(contact_vec):.6f}")


CENTER OF MASS & CONTACT VECTOR

Grain 1 COM: [91.46132198 97.79491779 85.28114454]
Grain 2 COM: [82.21752114 37.7540736  87.30044833]

Connection vector (COM2 - COM1): [ -9.24380084 -60.04084419   2.0193038 ]
Distance between COMs: 60.7818 Å

Normalized contact vector: [-0.1520817  -0.98780942  0.03322217]
Magnitude check: 1.000000


In [5]:
# Analyze contact vector alignment with Grain 1 axes
print("\n" + "="*80)
print("GRAIN 1 - Contact Vector Alignment Analysis")
print("="*80)

# Dot products with axes
dot_a1 = np.dot(contact_vec, a1)
dot_b1 = np.dot(contact_vec, b1)
dot_c1 = np.dot(contact_vec, c1)

print(f"\nDot products with axes:")
print(f"  contact_vec · a1 = {dot_a1:+.6f}")
print(f"  contact_vec · b1 = {dot_b1:+.6f}")
print(f"  contact_vec · c1 = {dot_c1:+.6f}")

# Angles with axes
angle_a1 = np.degrees(np.arccos(np.clip(np.abs(dot_a1), -1, 1)))
angle_b1 = np.degrees(np.arccos(np.clip(np.abs(dot_b1), -1, 1)))
angle_c1 = np.degrees(np.arccos(np.clip(np.abs(dot_c1), -1, 1)))

print(f"\nAngles with axes:")
print(f"  angle with a1: {angle_a1:.2f}°")
print(f"  angle with b1: {angle_b1:.2f}°")
print(f"  angle with c1: {angle_c1:.2f}°")

# Determine closest axis
angles_g1 = {'a': angle_a1, 'b': angle_b1, 'c': angle_c1}
closest_axis_g1 = min(angles_g1, key=angles_g1.get)
print(f"\n>>> Contact vector is most aligned with axis '{closest_axis_g1}' (angle: {angles_g1[closest_axis_g1]:.2f}°)")

# Contact plane determination for Grain 1
g1_plane = contactplan(a1, b1, c1, contact_vec)
print(f"\n>>> GRAIN 1 CONTACT PLANE: {g1_plane.upper()}")



GRAIN 1 - Contact Vector Alignment Analysis

Dot products with axes:
  contact_vec · a1 = +0.179801
  contact_vec · b1 = -0.695262
  contact_vec · c1 = +0.006425

Angles with axes:
  angle with a1: 79.64°
  angle with b1: 45.95°
  angle with c1: 89.63°

>>> Contact vector is most aligned with axis 'b' (angle: 45.95°)

>>> GRAIN 1 CONTACT PLANE: AC


In [6]:
# Analyze contact vector alignment with Grain 2 axes
print("\n" + "="*80)
print("GRAIN 2 - Contact Vector Alignment Analysis")
print("="*80)

# Dot products with axes
dot_a2 = np.dot(contact_vec, a2)
dot_b2 = np.dot(contact_vec, b2)
dot_c2 = np.dot(contact_vec, c2)

print(f"\nDot products with axes:")
print(f"  contact_vec · a2 = {dot_a2:+.6f}")
print(f"  contact_vec · b2 = {dot_b2:+.6f}")
print(f"  contact_vec · c2 = {dot_c2:+.6f}")

# Angles with axes
angle_a2 = np.degrees(np.arccos(np.clip(np.abs(dot_a2), -1, 1)))
angle_b2 = np.degrees(np.arccos(np.clip(np.abs(dot_b2), -1, 1)))
angle_c2 = np.degrees(np.arccos(np.clip(np.abs(dot_c2), -1, 1)))

print(f"\nAngles with axes:")
print(f"  angle with a2: {angle_a2:.2f}°")
print(f"  angle with b2: {angle_b2:.2f}°")
print(f"  angle with c2: {angle_c2:.2f}°")

# Determine closest axis
angles_g2 = {'a': angle_a2, 'b': angle_b2, 'c': angle_c2}
closest_axis_g2 = min(angles_g2, key=angles_g2.get)
print(f"\n>>> Contact vector is most aligned with axis '{closest_axis_g2}' (angle: {angles_g2[closest_axis_g2]:.2f}°)")

# Contact plane determination for Grain 2
g2_plane = contactplan(a2, b2, c2, contact_vec)
print(f"\n>>> GRAIN 2 CONTACT PLANE: {g2_plane.upper()}")

# %%
# Summary of contact plane analysis
print("\n" + "="*80)
print("CONTACT PLANE ANALYSIS SUMMARY")
print("="*80)
print(f"\nGrain 1 contact plane: {g1_plane.upper()}")
print(f"Grain 2 contact plane: {g2_plane.upper()}")
print(f"\nContact plane pair: {g1_plane}-{g2_plane}")
print(f"\nContact vector points from Grain 1 → Grain 2")
print(f"  Most aligned with Grain 1 axis: {closest_axis_g1} ({angles_g1[closest_axis_g1]:.2f}°)")
print(f"  Most aligned with Grain 2 axis: {closest_axis_g2} ({angles_g2[closest_axis_g2]:.2f}°)")



GRAIN 2 - Contact Vector Alignment Analysis

Dot products with axes:
  contact_vec · a2 = +0.233497
  contact_vec · b2 = -0.753691
  contact_vec · c2 = +0.196632

Angles with axes:
  angle with a2: 76.50°
  angle with b2: 41.09°
  angle with c2: 78.66°

>>> Contact vector is most aligned with axis 'b' (angle: 41.09°)

>>> GRAIN 2 CONTACT PLANE: AC

CONTACT PLANE ANALYSIS SUMMARY

Grain 1 contact plane: AC
Grain 2 contact plane: AC

Contact plane pair: ac-ac

Contact vector points from Grain 1 → Grain 2
  Most aligned with Grain 1 axis: b (45.95°)
  Most aligned with Grain 2 axis: b (41.09°)


In [7]:
"""
## 2. Misorientation Analysis
"""

# %%
print("\n\n" + "="*80)
print("MISORIENTATION ANALYSIS")
print("="*80)

# GB normal is the same as contact vector
gb_normal = contact_vec

print(f"\nGB Normal (same as contact vector): {gb_normal}")
print(f"Magnitude check: {np.linalg.norm(gb_normal):.6f}")

# %%
# Grain frames for misorientation
print("\n" + "-"*80)
print("Grain Frames (for misorientation calculation)")
print("-"*80)

print("\nGrain 1 frame matrix (columns = a1, b1, c1):")
g1_frame = np.column_stack([a1, b1, c1])
print(g1_frame)
print(f"Determinant: {np.linalg.det(g1_frame):.6f}")

print("\nGrain 2 frame matrix (columns = a2, b2, c2):")
g2_frame = np.column_stack([a2, b2, c2])
print(g2_frame)
print(f"Determinant: {np.linalg.det(g2_frame):.6f}")

# %%
# GB normal alignment with Grain 1 axes
print("\n" + "="*80)
print("GB NORMAL ALIGNMENT - GRAIN 1")
print("="*80)

print(f"\nDot products with Grain 1 axes:")
print(f"  GB_normal · a1 = {dot_a1:+.6f}  (angle: {angle_a1:.2f}°)")
print(f"  GB_normal · b1 = {dot_b1:+.6f}  (angle: {angle_b1:.2f}°)")
print(f"  GB_normal · c1 = {dot_c1:+.6f}  (angle: {angle_c1:.2f}°)")
print(f"\n>>> GB normal is most aligned with Grain 1 axis '{closest_axis_g1}' ({angles_g1[closest_axis_g1]:.2f}°)")

print("\n" + "="*80)
print("GB NORMAL ALIGNMENT - GRAIN 2")
print("="*80)

print(f"\nDot products with Grain 2 axes:")
print(f"  GB_normal · a2 = {dot_a2:+.6f}  (angle: {angle_a2:.2f}°)")
print(f"  GB_normal · b2 = {dot_b2:+.6f}  (angle: {angle_b2:.2f}°)")
print(f"  GB_normal · c2 = {dot_c2:+.6f}  (angle: {angle_c2:.2f}°)")
print(f"\n>>> GB normal is most aligned with Grain 2 axis '{closest_axis_g2}' ({angles_g2[closest_axis_g2]:.2f}°)")



MISORIENTATION ANALYSIS

GB Normal (same as contact vector): [-0.1520817  -0.98780942  0.03322217]
Magnitude check: 1.000000

--------------------------------------------------------------------------------
Grain Frames (for misorientation calculation)
--------------------------------------------------------------------------------

Grain 1 frame matrix (columns = a1, b1, c1):
[[-0.73108703 -0.41207105 -0.68208423]
 [-0.04657     0.7829571   0.07404042]
 [ 0.68069302  0.46602106 -0.72751571]]
Determinant: 0.813178

Grain 2 frame matrix (columns = a2, b2, c2):
[[-0.98735521 -0.52844688 -0.11731201]
 [-0.07975902  0.84027681 -0.21361546]
 [ 0.13699703 -0.12115597 -0.96984861]]
Determinant: 0.898903

GB NORMAL ALIGNMENT - GRAIN 1

Dot products with Grain 1 axes:
  GB_normal · a1 = +0.179801  (angle: 79.64°)
  GB_normal · b1 = -0.695262  (angle: 45.95°)
  GB_normal · c1 = +0.006425  (angle: 89.63°)

>>> GB normal is most aligned with Grain 1 axis 'b' (45.95°)

GB NORMAL ALIGNMENT - GRAIN

In [8]:
# Compute misorientation using the high-level function
print("\n" + "="*80)
print("COMPUTING MISORIENTATION (using orix)")
print("="*80)
print(f"Symmetry: {symmetry_name}")

try:
    miso = misorientation_for_group(
        g1_gro_file=g1_gro_file,
        g2_gro_file=g2_gro_file,
        g1_txt=g1_txt,
        g2_txt=g2_txt,
        symmetry_name=symmetry_name
    )
    
    print("\n" + "="*80)
    print("MISORIENTATION RESULTS")
    print("="*80)
    
    if miso['theta_deg'] is not None:
        print(f"\n✓ Misorientation angle (Θ):  {miso['theta_deg']:.4f}°")
        print(f"✓ Twist component:           {miso['twist_deg']:.4f}°")
        print(f"✓ Tilt component:            {miso['tilt_deg']:.4f}°")
        
        if miso['axis'] is not None:
            print(f"\nMisorientation axis (unit vector): {miso['axis']}")
            print(f"Axis magnitude check: {np.linalg.norm(miso['axis']):.6f}")
            
            # Analyze misorientation axis alignment
            axis_dot_gb = np.dot(miso['axis'], gb_normal)
            axis_angle_gb = np.degrees(np.arccos(np.clip(np.abs(axis_dot_gb), -1, 1)))
            print(f"\nMisorientation axis alignment with GB normal:")
            print(f"  axis · GB_normal = {axis_dot_gb:+.6f}")
            print(f"  angle = {axis_angle_gb:.2f}°")
        
        print(f"\nMethod used: {miso['method']}")
        
        # Verification
        if miso['twist_deg'] is not None and miso['tilt_deg'] is not None:
            reconstructed = np.sqrt(miso['twist_deg']**2 + miso['tilt_deg']**2)
            print(f"\nVerification: √(twist² + tilt²) = {reconstructed:.4f}°")
            print(f"              (should ≈ Θ = {miso['theta_deg']:.4f}°)")
            print(f"              Difference: {abs(reconstructed - miso['theta_deg']):.6f}°")
    else:
        print("\n⚠ Misorientation could not be computed.")
        print("  Check that orix is installed: pip install orix")
        
except Exception as e:
    print(f"\n❌ Error computing misorientation: {e}")
    import traceback
    traceback.print_exc()



COMPUTING MISORIENTATION (using orix)
Symmetry: triclinic

MISORIENTATION RESULTS

✓ Misorientation angle (Θ):  40.0262°
✓ Twist component:           38.4073°
✓ Tilt component:            11.2684°

Misorientation axis (unit vector): [-0.41173434 -0.90418123  0.11371518]
Axis magnitude check: 1.000000

Misorientation axis alignment with GB normal:
  axis · GB_normal = +0.959554
  angle = 16.35°

Method used: orix

Verification: √(twist² + tilt²) = 40.0262°
              (should ≈ Θ = 40.0262°)
              Difference: 0.000000°


In [9]:
# %% 
# ======================================================================
#   MISORIENTATION AXIS ALIGNMENT ANALYSIS 
# ======================================================================

if 'miso' in locals() and miso['axis'] is not None:
    mis_axis = miso['axis']   # unit vector

    print("\n" + "="*80)
    print("MISORIENTATION AXIS ALIGNMENT WITH GRAIN 1, GRAIN 2, & GB NORMAL")
    print("="*80)

    def angle_deg(u, v):
        """Return angle (deg) between two vectors."""
        dp = np.dot(u, v)
        dp = np.clip(dp, -1.0, 1.0)
        return np.degrees(np.arccos(np.abs(dp))), dp

    # -------------------------------
    # Alignment for Grain 1 axes
    # -------------------------------
    ang_a1, dot_a1m = angle_deg(mis_axis, a1)
    ang_b1, dot_b1m = angle_deg(mis_axis, b1)
    ang_c1, dot_c1m = angle_deg(mis_axis, c1)

    print("\nGRAIN 1:")
    print(f"  axis · a1 = {dot_a1m:+.6f}   → angle = {ang_a1:.2f}°")
    print(f"  axis · b1 = {dot_b1m:+.6f}   → angle = {ang_b1:.2f}°")
    print(f"  axis · c1 = {dot_c1m:+.6f}   → angle = {ang_c1:.2f}°")

    # -------------------------------
    # Alignment for Grain 2 axes
    # -------------------------------
    ang_a2, dot_a2m = angle_deg(mis_axis, a2)
    ang_b2, dot_b2m = angle_deg(mis_axis, b2)
    ang_c2, dot_c2m = angle_deg(mis_axis, c2)

    print("\nGRAIN 2:")
    print(f"  axis · a2 = {dot_a2m:+.6f}   → angle = {ang_a2:.2f}°")
    print(f"  axis · b2 = {dot_b2m:+.6f}   → angle = {ang_b2:.2f}°")
    print(f"  axis · c2 = {dot_c2m:+.6f}   → angle = {ang_c2:.2f}°")

    # -------------------------------
    # Alignment with GB normal
    # -------------------------------
    ang_gb, dot_gb = angle_deg(mis_axis, gb_normal)

    print("\nGB NORMAL:")
    print(f"  axis · GB_normal = {dot_gb:+.6f}   → angle = {ang_gb:.2f}°")

    # -------------------------------
    # Summary (closest axis)
    # -------------------------------
    all_angles = {
        "G1_a": ang_a1, "G1_b": ang_b1, "G1_c": ang_c1,
        "G2_a": ang_a2, "G2_b": ang_b2, "G2_c": ang_c2,
        "GB_normal": ang_gb
    }

    closest = min(all_angles, key=all_angles.get)

    print("\n" + "-"*80)
    print(f"Closest alignment: {closest}  (angle = {all_angles[closest]:.2f}°)")
    print("-"*80)

else:
    print("\n⚠ No misorientation axis found, cannot compute alignment.")



MISORIENTATION AXIS ALIGNMENT WITH GRAIN 1, GRAIN 2, & GB NORMAL

GRAIN 1:
  axis · a1 = +0.420526   → angle = 65.13°
  axis · b1 = -0.485278   → angle = 60.97°
  axis · c1 = +0.131162   → angle = 82.46°

GRAIN 2:
  axis · a2 = +0.494223   → angle = 60.38°
  axis · b2 = -0.555960   → angle = 56.22°
  axis · c2 = +0.131162   → angle = 82.46°

GB NORMAL:
  axis · GB_normal = +0.959554   → angle = 16.35°

--------------------------------------------------------------------------------
Closest alignment: GB_normal  (angle = 16.35°)
--------------------------------------------------------------------------------


In [10]:
"""
## Complete Summary
"""

# %%
print("\n\n" + "#"*80)
print("#" + " "*78 + "#")
print("#" + " "*25 + "COMPLETE SUMMARY" + " "*37 + "#")
print("#" + " "*78 + "#")
print("#"*80)

print("\n" + "="*80)
print("GRAIN INFORMATION")
print("="*80)
print(f"Grain 1: {g1_gro_file}")
print(f"  Latvec file: {g1_txt}")
print(f"  COM: [{com1[0]:.4f}, {com1[1]:.4f}, {com1[2]:.4f}]")
print(f"  Contact plane: {g1_plane.upper()}")

print(f"\nGrain 2: {g2_gro_file}")
print(f"  Latvec file: {g2_txt}")
print(f"  COM: [{com2[0]:.4f}, {com2[1]:.4f}, {com2[2]:.4f}]")
print(f"  Contact plane: {g2_plane.upper()}")

print("\n" + "="*80)
print("CONTACT ANALYSIS")
print("="*80)
print(f"Contact vector: [{contact_vec[0]:+.6f}, {contact_vec[1]:+.6f}, {contact_vec[2]:+.6f}]")
print(f"Distance between grains: {np.linalg.norm(conn_vec):.4f} Å")
print(f"Contact plane pair: {g1_plane}-{g2_plane}")
print("Contact vector alignment:")
print(f"  → Grain 1: closest to {closest_axis_g1}-axis ({angles_g1[closest_axis_g1]:.2f}°)")
print(f"  → Grain 2: closest to {closest_axis_g2}-axis ({angles_g2[closest_axis_g2]:.2f}°)")

if 'miso' in locals() and miso['theta_deg'] is not None:
    print("\n" + "="*80)
    print("MISORIENTATION RESULTS")
    print("="*80)
    print(f"Symmetry: {symmetry_name}")
    print(f"\nMisorientation angle (Θ): {miso['theta_deg']:.4f}°")
    print(f"Twist component:          {miso['twist_deg']:.4f}°")
    print(f"Tilt component:           {miso['tilt_deg']:.4f}°")
    if miso['axis'] is not None:
        print(f"\nMisorientation axis: [{miso['axis'][0]:+.6f}, {miso['axis'][1]:+.6f}, {miso['axis'][2]:+.6f}]")
        axis_dot_gb = np.dot(miso['axis'], gb_normal)
        axis_angle_gb = np.degrees(np.arccos(np.clip(np.abs(axis_dot_gb), -1, 1)))
        print(f"Axis alignment with GB normal: {axis_angle_gb:.2f}°")

print("\n" + "#"*80)
print("Analysis complete!")
print("#"*80)




################################################################################
#                                                                              #
#                         COMPLETE SUMMARY                                     #
#                                                                              #
################################################################################

GRAIN INFORMATION
Grain 1: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g1.gro
  Latvec file: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g1_latvecs.txt
  COM: [91.4613, 97.7949, 85.2811]
  Contact plane: AC

Grain 2: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g2.gro
  Latvec file: /home/sgarg/structural_analysis_gb/test_b45/slab_01/slab_01_seg02_y67.57_g2_latvecs.txt
  COM: [82.2175, 37.7541, 87.3004]
  Contact plane: AC

CONTACT ANALYSIS
Contact vector: [-0.152082, -0.987809, +0.033222]
Distance be

In [12]:
# %%
# Misorientation axis components in Grain 1 and Grain 2 bases

mis_axis = np.array(miso["axis"])  # ensure np.array

# components in grain-1 basis
coeff_g1 = g1_frame.T @ mis_axis
# components in grain-2 basis
coeff_g2 = g2_frame.T @ mis_axis

print("\nMisorientation axis components in Grain 1 basis (a1,b1,c1):")
print(f"  n = {coeff_g1[0]:+.3f} a1  + {coeff_g1[1]:+.3f} b1  + {coeff_g1[2]:+.3f} c1")

print("\nMisorientation axis components in Grain 2 basis (a2,b2,c2):")
print(f"  n = {coeff_g2[0]:+.3f} a2  + {coeff_g2[1]:+.3f} b2  + {coeff_g2[2]:+.3f} c2")



Misorientation axis components in Grain 1 basis (a1,b1,c1):
  n = +0.421 a1  + -0.485 b1  + +0.131 c1

Misorientation axis components in Grain 2 basis (a2,b2,c2):
  n = +0.494 a2  + -0.556 b2  + +0.131 c2
